# 00 - Introduction: the question, the data, the plan

Read this first, then work through the numbered notebooks in order.

If you are new to time-series ML or seeing this dataset for the first time,
every markdown block explains why we are doing what the code does, not just
what the code does.

## The problem

Jet engines are expensive to run and expensive to fail. Airlines want to
detect the warning signs of an engine going bad before it grounds a plane.
This is the domain of *predictive maintenance*: use sensor readings to
predict trouble early enough that maintenance can be scheduled instead of
forced.

The approach used here is called *onset-of-degradation detection*: learn
what a healthy engine looks like, then watch each engine over time and
raise an alarm when its readings start drifting away from healthy.

## The dataset

**C-MAPSS FD004** (NASA Ames, 2008) is a simulated jet-engine degradation
benchmark. It contains 249 engines in the training set and 248 in the test
set. Each engine is monitored cycle-by-cycle across 21 sensors plus 3
operational settings (altitude, Mach number, throttle) until failure. FD004
is the hardest of the four C-MAPSS subsets because it mixes 6 flight
conditions and 2 failure modes. Any anomaly detected must be
distinguishable from perfectly normal regime changes.

Files in `data/`:

- `train_FD004.txt`: full run-to-failure trajectories, one row per (engine, cycle).
- `test_FD004.txt`: trajectories cut off partway through life.
- `RUL_FD004.txt`: true remaining useful life at the cut, one value per test engine.

## What "anomaly" means here (framing)

C-MAPSS does not ship anomaly labels. Every engine eventually fails, so the
notion of "which point is anomalous?" is up to us. Three reasonable framings
exist. The one used here:

> **Framing (a), onset of degradation.** The first ~30-50% of each engine's
> life is treated as healthy. Unsupervised detectors are trained on those
> healthy cycles only. Any cycle whose feature vector looks unlike the
> healthy distribution is flagged as anomalous. Success is measured by
> *lead time*: how many cycles before failure did the first flag fire?

That framing lets us use two clean, unsupervised methods that need no
labels. The whole point of the exercise is to learn what "normal" looks
like without being told when things go wrong.

## Roadmap through the notebooks

| # | Notebook | What you will learn |
|---|----------|---------------------|
| 00 | this one | The problem, dataset, framing |
| 01 | `01_load_and_eda` | pandas basics, loading and first-pass exploration, spotting dead sensors, identifying the 6 operating regimes |
| 02 | `02_feature_engineering` | Hypothesis-driven feature design. For every feature we state the hypothesis, do the math, and validate with a plot |
| 03 | `03_train_iforest` | Isolation Forest: how it works, training on healthy-only data, picking a threshold |
| 04 | `04_train_dbscan` | DBSCAN: density-based clustering as a very different second opinion; eps picked via k-distance plot |
| 05 | `05_evaluate` | The lead-time metric, precision and recall at a threshold, comparing the two detectors' agreement |
| 06 | `06_visuals` | Summary plots that make the story readable at a glance |

## Two detectors, on purpose

Two very different approaches are used on the same features:

- **Isolation Forest**: an ensemble of random trees. It scores each point
  by how easy it is to isolate. Anomalies isolate quickly (few splits).
- **DBSCAN**: a density-based clustering algorithm. Points that do not
  belong to any dense cluster are labelled *noise* (`-1`). Noise = anomaly.

When two methods that make completely different assumptions agree that a
particular cycle is anomalous, that agreement is strong evidence. When they
disagree, we learn something too.

## Prerequisites

Python 3.10+ and the packages in `../requirements.txt`. From the repo root:

```bash
python -m venv venv
source venv/Scripts/activate   # macOS/Linux: source venv/bin/activate
pip install -r requirements.txt
```

Then start Jupyter from the repo root so the `src/` imports resolve:

```bash
jupyter lab
```

Environment sanity-check:

In [ ]:
import sys, platform
print("Python :", sys.version.split()[0])
print("System :", platform.system(), platform.release())

import numpy, pandas, sklearn, matplotlib, seaborn
for pkg in (numpy, pandas, sklearn, matplotlib, seaborn):
    print(f"{pkg.__name__:12s} {pkg.__version__}")

## Next

Open **`01_load_and_eda.ipynb`**. It loads FD004 for the first time, walks
through what each column means, and shows the plots that will inform every
feature we design in notebook 02.